In [30]:
!pwd
!ls data


/Users/yevamykhailovska/Documents/backup/mcbiclust
CCLE_RNAseq_genes_rpkm_20180929.gct
E_coli_v4_Build_6_chips907probes7459.tab
ecocyc.gaf
Human.MitoCarta3.0.csv
Human.MitoCarta3.0.xls


# CCLE Single Run without fit and summary

In [ ]:

# # ═══════════════════════════════════════════════════════════════
# MCbiclust Python — Usage Example
# Mirrors the R package vignette (Bentham et al. 2017)
# ═══════════════════════════════════════════════════════════════

import numpy as np
import sys
sys.path.insert(0, "src")

from data_loading import load_ccle_raw, load_ecoli, ExpressionDataset, load_ccle_mitocarta
from preprocessing import log2_transform, select_genes, filter_low_expression_genes
from correlation import avg_abs_corr_rows
from gene_seed import select_initial_seed_genes
from find_seed_sample import find_seed_bicluster
from pruning import prune_bicluster_genes
from sample_sort import extend_bicluster_samples_fast, plot_sample_extension_summary
from correlation_vector import cv_eval, print_top_genes, gene_vec_fun
from pca import pc1_vec_fun, plot_fork, pc1_summary_stats
from fork import  fork_classifier, print_fork_summary, plot_fork_classified
from thresholding import threshold_bic, plot_threshold_summary, pc1_align
from silhouette import silhouette_clust_groups, plot_silhouette, average_corvec_per_cluster
from multi_run import run_multiple, multi_sample_sort_prep
from mcbiclust import MCbiclust, MCbiclustMulti
from enrichment import go_enrichment_pipeline, fetch_go_term_names
from point_score import point_score_calc
from cv_plot import cv_plot, plot_correlation_vector_distribution, plot_correlation_vector_ranked
from seed_plots import plot_alpha_vs_iterations, plot_sample_corr_heatmap, plot_gene_corr_heatmap

import pandas as pd
import numpy as np



# data loading, Mitocarta
ccle_log, gene_set, ccle_mito, mitocarta_genes = load_ccle_mitocarta(ccle_path="data/CCLE_RNAseq_genes_rpkm_20180929.gct",
    mitocarta_path="data/Human.MitoCarta3.0.xls")

ccle_mito.summary()



# Select initial gene set (use all mitochondrial genes)
gene_set = select_initial_seed_genes(
    X                   = ccle_mito.X,
    n_genes             = 1000,
    biological_gene_set = np.arange(ccle_mito.n_genes),
    random_state        = 42,
)

# Compare random seed vs optimised seed
rng = np.random.default_rng(103)
random_seed = rng.choice(ccle_mito.n_samples, size=10, replace=False)
random_alpha = avg_abs_corr_rows(ccle_mito.X[gene_set][:, random_seed])
print(f"Random seed alpha:    {random_alpha:.4f}")

# FindSeed — greedy stochastic search for best 10 samples
seed, seed_alpha, improvements, history, history_iterations = find_seed_bicluster(
    X=ccle_mito.X,
    gene_set=gene_set,
    n_samples=10,
    iterations=1000,
    random_state=102,
    log_improvements=True,
    print_every_improvement=False,
    print_every_iter=False,
    verbose=False,
)

plot_alpha_vs_iterations(
    steps=history_iterations,
    alpha_history=history,
    title="FindSeed: Alpha Convergence"
)

plot_sample_corr_heatmap(ccle_mito.X, subset_size=10, seed=42)

plot_gene_corr_heatmap(
    X=ccle_mito.X,
    sample_set=seed,
    gene_names=ccle_mito.genes,
    title="FindSeed gene-gene correlation",
    max_genes=700,
    cmap="hot",
    vmin=-1,
    vmax=1,
    figsize=(18, 10),
)


kept_local_idx, pruned_genes, cluster_labels, hi_cor_values, group_alphas = \
    prune_bicluster_genes(
        X                     = ccle_mito.X,
        gene_set              = gene_set,
        sample_set            = seed,
        n_groups              = 8,
        plot_dendrogram       = False,
        dendrogram_sample_size= 100,
    )

print(f"Genes before pruning: {len(gene_set)}")
print(f"Genes after pruning:  {len(pruned_genes)}")

plot_gene_corr_heatmap(
    X=ccle_mito.X[pruned_genes],
    sample_set=seed,
    gene_names=ccle_mito.genes[pruned_genes],
    title="Pruned gene-gene correlation",
    max_genes=700,
    cmap="hot",
    vmin=-1,
    vmax=1,
    figsize=(18, 10),
)

corr_vec, best_idx = cv_eval(
    X_part=ccle_mito.X[pruned_genes], 
    X_all=ccle_log.X,
    seed=seed,
    splits=10,
    gene_names=ccle_log.genes
)


results_df = go_enrichment_pipeline(
    cor_vec=corr_vec,
    gene_names=ccle_log.genes,
    gaf_path="data/goa_human.gaf",
    top_n=10,
    min_genes=10,
    alternative="greater",
)

# all_go_ids = results_df["go_id"].tolist()
# go_term_names = fetch_go_term_names(all_go_ids)
# results_df["TERM"] = results_df["go_id"].map(go_term_names)
# results_df.to_csv("go_enrichment_results.csv", index=False)

results_df = pd.read_csv("go_enrichment_results.csv")

specific_go = results_df[
    (results_df["num.genes"] >= 10) &
    (results_df["num.genes"] <= 500)
].copy()

specific_go = specific_go.sort_values("CV.av.value", ascending=False)
specific_go.head(20)


ranked_samples, sample_alphas, picked = extend_bicluster_samples_fast(
    X                       = ccle_mito.X,
    gene_set                = pruned_genes,
    initial_sample_set      = seed,
    progress_every          = 50,
    print_only_improvements = False,
    max_rank                = None,
)

full_alpha = [np.nan] * len(ranked_samples)

# first 10 are seed, so no extension alpha there
for i, a in enumerate(sample_alphas, start=len(seed)):
    full_alpha[i] = a

df_samples = pd.DataFrame({
    "rank": np.arange(1, len(ranked_samples) + 1),
    "sample_index": ranked_samples,
    "alpha_after_addition": full_alpha
})

df_samples.to_csv("sample_order.csv", index=False)

plot_sample_extension_summary(
    X              = ccle_mito.X,
    gene_set       = pruned_genes,
    ranked_samples = ranked_samples,
    sample_alphas  = sample_alphas,
    sample_names   = ccle_mito.samples,
    n_seed         = 10,
)


pc1_vec = pc1_vec_fun(
    X=ccle_mito.X,
    gene_set=pruned_genes,
    seed_sort=ranked_samples,
    n=10,
    align_sign=False   
)

pc1_summary_stats(pc1_vec, n_seed=10)
plot_fork(pc1_vec, n_seed=10, title="Fork Plot")

bic_genes, bic_samps = threshold_bic(
    cor_vec=corr_vec,
    sort_order=ranked_samples,
    pc1=pc1_vec,
    samp_sig=0.05,
    sample_names=ccle_mito.samples
)

pc1_vec = pc1_align(
    gem=ccle_log.X,
    pc1=pc1_vec,
    sort_order=ranked_samples,
    cor_vec=corr_vec,
    bic=(bic_genes, bic_samps)
)

plot_threshold_summary(
    cor_vec=corr_vec,
    pc1=pc1_vec,
    bic_genes=bic_genes,
    bic_samps=bic_samps,
    sort_order=ranked_samples,
    sample_names=ccle_mito.samples
)


fork_status = fork_classifier(
    pc1=pc1_vec,
    samp_num=len(bic_samps)
)

print_fork_summary(
    pc1=pc1_vec,
    fork_status=fork_status,
    sort_order=ranked_samples,
    bic_samps=bic_samps,
    bic_genes=bic_genes,
    sample_names=ccle_mito.samples,
    n_top=20
)

plot_fork_classified(
    pc1=pc1_vec,
    fork_status=fork_status,
    sort_order=ranked_samples,
    bic_samps=bic_samps,
    n_seed=10
)



from metadata_analysis import (
    load_ccle_sample_info,
    build_sample_results_dataframe,
    merge_sample_metadata,
    collapse_rare_categories,
    plot_pc1_by_category,
    chi_square_fork_vs_category,
)

from fork_analysis import (
    split_samples_by_fork,
    compute_fork_expression_difference,
    get_top_fork_genes,
    run_fork_go_enrichment,
)

# metadata
CCLE_samples = load_ccle_sample_info("data/sample_info.csv")

CCLE_df = build_sample_results_dataframe(
    ranked_samples=ranked_samples,
    sample_names=ccle_log.samples,
    pc1_vec=pc1_vec,
    fork_status=fork_status,
)

CCLE_df_samples = merge_sample_metadata(CCLE_df, CCLE_samples)

CCLE_df_samples = collapse_rare_categories(
    CCLE_df_samples,
    category_col="sample_collection_site",
    min_count=15,
    other_label="Other",
    new_col="site2",
)

import matplotlib.pyplot as plt
plt.style.use("ggplot")

# Site.Primary
plot_pc1_by_category(
    CCLE_df_samples,
    category_col="sample_collection_site",
    title="PC1 by sample collection site"
)

# Site.Primary2
plot_pc1_by_category(
    CCLE_df_samples,
    category_col="sample_collection_site",
    min_count=15,
    other_label="Other",
    title="PC1 by sample collection site (rare groups collapsed)"
)

# Gender
plot_pc1_by_category(
    CCLE_df_samples,
    category_col="sex",
    title="PC1 by sex"
)

site_test = chi_square_fork_vs_category(CCLE_df_samples, "site2")
gender_test = chi_square_fork_vs_category(CCLE_df_samples, "sex", exclude_values=["U"])

print("Site p-value:", site_test["p_value"])
print("Gender p-value:", gender_test["p_value"])



In [43]:
!pwd

/Users/yevamykhailovska/Documents/backup/mcbiclust


# CCLE using fix(),  summary() methods

In [ ]:
import numpy as np
import sys
sys.path.insert(0, "src")

from data_loading import load_ccle_raw, load_ecoli, ExpressionDataset, load_ccle_mitocarta
from preprocessing import log2_transform, select_genes, filter_low_expression_genes
from correlation import avg_abs_corr_rows
from gene_seed import select_initial_seed_genes
from find_seed_sample import find_seed_bicluster
from pruning import prune_bicluster_genes
from sample_sort import extend_bicluster_samples_fast, plot_sample_extension_summary
from correlation_vector import cv_eval, print_top_genes, gene_vec_fun
from pca import pc1_vec_fun, plot_fork,  pc1_summary_stats
from fork import  fork_classifier, print_fork_summary, plot_fork_classified
from thresholding import threshold_bic, plot_threshold_summary
from silhouette import silhouette_clust_groups, plot_silhouette, average_corvec_per_cluster
from multi_run import run_multiple, multi_sample_sort_prep
from mcbiclust import MCbiclust, MCbiclustMulti
from enrichment import go_enrichment_pipeline, fetch_go_term_names
from point_score import point_score_calc
from cv_plot import cv_plot, plot_correlation_vector_distribution, plot_correlation_vector_ranked
from seed_plots import plot_alpha_vs_iterations, plot_sample_corr_heatmap, plot_gene_corr_heatmap

import pandas as pd
import numpy as np

ccle_log, gene_set, ccle_mito, mitocarta_genes = load_ccle_mitocarta(
    ccle_path="data/CCLE_RNAseq_genes_rpkm_20180929.gct",
    mitocarta_path="data/Human.MitoCarta3.0.xls"
)

mc = MCbiclust(
    n_samples=10,
    iterations=1000,
    n_genes=1000,
    splits=8,
    samp_sig=0.05,   
    random_state=42
)

mc.fit(ccle_mito)

mc.summary()

In [ ]:
from metadata_analysis import load_ccle_sample_info, chi_square_fork_vs_category
import matplotlib.pyplot as plt


CCLE_df = mc.to_sample_dataframe(ccle_log)
CCLE_df.head()
CCLE_samples = load_ccle_sample_info("data/sample_info.csv")

CCLE_df_samples = mc.merge_sample_metadata(
    dataset=ccle_log,
    metadata_df=CCLE_samples
)

plt.style.use("ggplot")

# Site.Primary
plot_pc1_by_category(
    CCLE_df_samples,
    category_col="sample_collection_site",
    title="PC1 by sample collection site"
)


# Gender
plot_pc1_by_category(
    CCLE_df_samples,
    category_col="sex",
    title="PC1 by sex"
)


go_results = mc.run_go_enrichment(
    gene_names=ccle_mito.genes,
    gaf_path="data/goa_human.gaf",
    top_n=10
)

go_results.head()

print(go_results["pval"].head())


# CCLE multi-run

In [ ]:
mcm = MCbiclustMulti(
    n_runs=100,
    n_genes=500,
    n_samples=10,
    iterations=500,
    splits=8,
    samp_sig=0.05,
    max_clusters=5,
    top_genes_num=300
)

mcm.fit(ccle_mito)
mcm.summary()


In [ ]:
cor_vec_mat, seeds_list = run_multiple(
    X=ccle_mito.X,
    n_runs=100,
    n_genes=500,
    n_samples=10,
    iterations=500,
    splits=8,
    cv_splits=10,
)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --------------------------------------------------
# 1. Silhouette clustering
# --------------------------------------------------
cluster_groups = silhouette_clust_groups(
    cor_vec_mat=cor_vec_mat,
    max_clusters=5,
    plots=True
)

plot_silhouette(
    cor_vec_mat=cor_vec_mat,
    cluster_groups=cluster_groups,
    title="Silhouette plot of biclusters"
)

# --------------------------------------------------
# 2. Average correlation vectors
# IMPORTANT:
# average_corvec should be the SIGNED average vector per cluster
# --------------------------------------------------
average_corvec, average_corvec_abs = average_corvec_per_cluster(
    cor_vec_mat,
    cluster_groups
)

# --------------------------------------------------
# 3. CV plot
# Replace mito_loc with your actual mitochondrial gene indices if available
# --------------------------------------------------
mito_loc = np.arange(ccle_mito.X.shape[0])

cv_plot(
    cor_vec_list=average_corvec,
    geneset_loc=mito_loc,
    geneset_name="Mitochondrial",
    alpha1=0.1,
    cnames=[f"R{i+1}" for i in range(len(average_corvec))]
)

# --------------------------------------------------
# 4. MultiSampleSortPrep
# --------------------------------------------------
top_mats, top_seeds = multi_sample_sort_prep(
    X=ccle_mito.X,
    av_corvec=average_corvec,
    top_genes_num=750,
    groups=cluster_groups,
    initial_seeds=seeds_list
)

# --------------------------------------------------
# 5. SampleSort for each bicluster
# THIS was missing
# --------------------------------------------------
samp_multi_sort = []
sample_alpha_multi = []
picked_multi = []

for i in range(len(top_mats)):
    print(f"\nRunning SampleSort for bicluster {i+1}...")

    ranked_i, alphas_i, picked_i = extend_bicluster_samples_fast(
        X=top_mats[i],
        gene_set=np.arange(top_mats[i].shape[0]),
        initial_sample_set=top_seeds[i],
        max_rank=None,
    )

    samp_multi_sort.append(ranked_i)
    sample_alpha_multi.append(alphas_i)
    picked_multi.append(picked_i)

# --------------------------------------------------
# 6. PC1 values
# --------------------------------------------------
pc1_vec_multi = []

for i in range(len(top_mats)):
    pc1_i = pc1_vec_fun(
        X=top_mats[i],
        gene_set=np.arange(top_mats[i].shape[0]),
        seed_sort=samp_multi_sort[i],
        n=10,
        align_sign=False,
    )
    pc1_vec_multi.append(pc1_i)

# --------------------------------------------------
# 7. Threshold biclusters
# --------------------------------------------------
bic_multi = []

for i in range(len(average_corvec)):
    bic_genes_i, bic_samps_i = threshold_bic(
        cor_vec=average_corvec[i],
        sort_order=samp_multi_sort[i],
        pc1=pc1_vec_multi[i],
        samp_sig=0.05,
        sample_names=ccle_mito.samples,
    )
    bic_multi.append((bic_genes_i, bic_samps_i))

# --------------------------------------------------
# 8. Align PC1
# --------------------------------------------------
for i in range(len(pc1_vec_multi)):
    pc1_vec_multi[i] = pc1_align(
        gem=ccle_mito.X,
        pc1=pc1_vec_multi[i],
        sort_order=samp_multi_sort[i],
        cor_vec=average_corvec[i],
        bic=bic_multi[i],
    )

# --------------------------------------------------
# 9. Build output dataframe
# Your result has only 1 bicluster, so do not hardcode Bic2
# --------------------------------------------------
multi_df = pd.DataFrame({
    "CCLE_name": ccle_mito.samples,
    "Bic1_order": np.argsort(samp_multi_sort[0]) + 1,
    "Bic1_PC1": pc1_vec_multi[0][np.argsort(samp_multi_sort[0])],
})

# --------------------------------------------------
# 10. Merge metadata
# --------------------------------------------------
CCLE_samples = load_ccle_sample_info("data/sample_info.csv")

multi_df_samples = multi_df.merge(
    CCLE_samples,
    on="CCLE_name",
    how="inner"
)

multi_df_samples = collapse_rare_categories(
    multi_df_samples,
    category_col="sample_collection_site",
    min_count=15,
    other_label="Other",
    new_col="site2",
)

# --------------------------------------------------
# 11. Plot fork for Bic1
# --------------------------------------------------
plt.figure(figsize=(12, 5))
for site, subdf in multi_df_samples.groupby("site2"):
    plt.scatter(
        subdf["Bic1_order"],
        subdf["Bic1_PC1"],
        s=20,
        alpha=0.8,
        label=site
    )

plt.xlabel("Bic1 order")
plt.ylabel("Bic1 PC1")
plt.title("Bic1 PC1 by sample collection site")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()